<a href="https://colab.research.google.com/github/ashuyad/learning-transformers/blob/main/notebooks/fine_tune_masked_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoModelForMaskedLM

checkpoint = "distilbert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(checkpoint)

In [ ]:
model.num_parameters()

In [ ]:
text = "This is a great [MASK]."

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
import torch
inputs = tokenizer(text, return_tensors = "pt")
token_logits = model(**inputs).logits
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = token_logits[0, mask_token_index, :]

top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

for token in top_5_tokens:
    print(f"'>>> {text.replace(tokenizer.mask_token, tokenizer.decode([token]))}'")

In [ ]:
from datasets import load_dataset

imdb_dataset = load_dataset("stanfordnlp/imdb")
imdb_dataset

In [ ]:
sample = imdb_dataset["train"].shuffle(seed=42).select(range(3))

for row in sample:
  print(f"\n>>> Review: {row['text']}")
  print(f"\n>>> Label: {row['label']}")

In [ ]:
def tokenize_function(example):
  result = tokenizer(example["text"])
  if tokenizer.is_fast:
    result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
  return result

In [ ]:
tokenized_dataset = imdb_dataset.map(tokenize_function, batched=True, remove_columns= ["text", "label"])

tokenized_dataset

In [ ]:
tokenized_dataset["train"][0]

In [ ]:
tokenizer.model_max_length

In [ ]:
chunk_size = 128

In [ ]:
tokenized_samples = tokenized_dataset["train"][:3]

for idx, sample in enumerate(tokenized_samples["input_ids"]):
  print(f"Review {idx} length: {len(sample)}")

In [ ]:
tokenized_samples.keys()

In [ ]:
concatenate_example = {
    k: sum(tokenized_samples[k], []) for k in tokenized_samples.keys()
}
total_length = len(concatenate_example["input_ids"])
print(f"'>>> Concatenated reviews length: {total_length}'")

In [ ]:
chunks = {
    k: [t[i:i+chunk_size] for i in range(0, total_length, chunk_size)] for k,t in concatenate_example.items()

}

for chunk in chunks["input_ids"]:
    print(f"'>>> Chunk length: {len(chunk)}'")

In [ ]:
def group_text(example):
  concatenate_example = { k: sum(example[k], []) for k in example.keys()}

  total_length = len(concatenate_example[list(example.keys())[0]])

  total_lenght = (total_length // chunk_size) * chunk_size

  result = {
      k: [t[i: i+chunk_size] for i in range (0, total_length, chunk_size)] for k, t in concatenate_example.items()
  }

  result["labels"] = result["input_ids"].copy()

  return result


In [ ]:
lm_dataset = tokenized_dataset.map(group_text, batched=True)
lm_dataset

In [ ]:
tokenizer.decode(lm_dataset["train"][1]["input_ids"])

In [ ]:
tokenizer.decode(lm_dataset["train"][1]["labels"])

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

In [ ]:
samples = [lm_dataset["train"][i] for i in range(2)]

for sample in samples:
  _ = sample.pop("word_ids")

for chunk in data_collator(samples)["input_ids"]:
  print(f">>>{tokenizer.decode(chunk)}")

In [ ]:
train_size = 10_000
test_size = int(0.1 * train_size)

downsampled_dataset = lm_dataset["train"].train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)
downsampled_dataset

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import TrainingArguments

batch_size = 64
# Show the training loss with every epoch
logging_steps = len(downsampled_dataset["train"]) // batch_size
model_name = checkpoint.split("/")[-1]

training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-imdb-smallset",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    push_to_hub=True,
    fp16=True,
    logging_steps=logging_steps,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
import math

eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")